# MACE

In [1]:
from ase import units
from ase.md.langevin import Langevin
from ase.io import read, write
import numpy as np
import time

from mace.calculators import MACECalculator, mace_polar

/opt/homebrew/Caskroom/miniforge/base/envs/mlff/lib/python3.13/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


### Download pretrained model's weight

- For MACE Polar:

    Go to https://github.com/ACEsuit/mace-foundations/releases and download, e.g., `MACE-POLAR-1-M.model`

---

In [2]:
calculator = MACECalculator(model_paths='MACE-POLAR-1-M.model', device='cpu')
init_conf = read('methylamine.xyz', '0')
init_conf.set_calculator(calculator)

# dyn = Langevin(init_conf, 0.5*units.fs, temperature_K=310, friction=5e-3)
# def write_frame():
#         dyn.atoms.write('md_3bpa.xyz', append=True)
# dyn.attach(write_frame, interval=50)
# dyn.run(100)
# print("MD finished!")

/opt/homebrew/Caskroom/miniforge/base/envs/mlff/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/var/folders/h0/6hz9lfgx7zx0wp8y0jf80xzr0000gn/T/ipykernel_75834/418621242.py:3: FutureWarning: Please use atoms.calc = calc
  init_conf.set_calculator(calculator)


In [3]:
calc = mace_polar(
    model="polar-1-m",        # or "polar-1-l"
    # device="cuda",
    default_dtype="float64",  # use float32 for faster MD
)

atoms = read("methylamine.xyz", index=0)
atoms.info["charge"] = 0
atoms.info["spin"] = 1 # Number of unpaired electrons + 1
atoms.info["external_field"] = [0.0, 0.0, 0.0]
atoms.calc = calc

energy = atoms.get_potential_energy()
forces = atoms.get_forces()

# extra polar outputs
mu       = calc.results["dipole"]                # total molecular dipole
rho      = calc.results["density_coefficients"]  # per-atom multipole coefficients
rho_spin = calc.results["spin_charge_density"]   # spin-resolved density
charges  = calc.results["charges"]               # per-atom monopoles

/opt/homebrew/Caskroom/miniforge/base/envs/mlff/lib/python3.13/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using MACE-Polar model for MACECalculator with /Users/nutt/.cache/mace/MACEPOLAR1Mmodel


In [4]:
print("Energy:", energy)
print("Forces:", forces)
print("Dipole:", mu)
print("Density Coefficients:", rho)
print("Spin Charge Density:", rho_spin)
print("Charges:", charges)

Energy: -2607.89420452812
Forces: [[-0.93773546 -0.31879425 -0.08497359]
 [-0.08103065 -0.09407374 -0.01572411]
 [ 0.3625025   0.17505331  0.43014814]
 [ 0.4115034   0.2525898  -0.34431761]
 [ 0.11879497  0.0974911  -0.24064743]
 [ 0.08785998  0.04684216  0.26868403]
 [ 0.03810526 -0.15910838 -0.01316944]]
Dipole: [-0.0029417   0.27582139  0.02667524]
Density Coefficients: [[-0.02985179  0.20437551  0.01640528 -0.05998178]
 [-0.0039454  -0.01500108 -0.0038406  -0.03876796]
 [ 0.01038469  0.02149321  0.17447901  0.08605829]
 [ 0.0104134   0.05384734 -0.15550347  0.10610757]
 [ 0.0045923  -0.04259336  0.06126324 -0.0424462 ]
 [ 0.00460886 -0.02927687 -0.07310105 -0.03427832]
 [ 0.00379794  0.07132454  0.00637468 -0.01133011]]
Spin Charge Density: [[[ 0.06253864  0.6447571   0.041312   -0.36182464]
  [-0.09239044 -0.44038159 -0.02490672  0.30184286]]

 [[ 0.02493186  0.06727567  0.02683819  0.32903832]
  [-0.02887726 -0.08227675 -0.03067879 -0.36780628]]

 [[-0.02592208 -0.07468755 -0.371